# Run DETR model
* Use frames provided by Heba in the google drive.
* Data folder ~/scratch/2026_freeman_bird_behavior/data/wz_data (H23 or H03)
* The script of base DETR model was cloned from [DETR](https://github.com/facebookresearch/detr)

In [1]:
import sys
sys.path.append('/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior')

from detr_detector.models.detr import DETR
from detr_detector.models.backbone import build_backbone
from detr_detector.models.transformer import build_transformer
from detr_detector.engine import train_one_epoch, evaluate

# 1. Verify GPU

In [2]:
import torch
print(f"# GPU available:  {torch.cuda.is_available()}")
print(f"# GPU name:       {torch.cuda.get_device_name(0)}")
print(f"# VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# GPU available:  True
# GPU name:       NVIDIA H100 80GB HBM3
# VRAM:           85.0 GB


In [3]:
# import torch
# print(f"# GPU available:  {torch.cuda.is_available()}")
# print(f"# GPU name:       {torch.cuda.get_device_name(0)}")
# print(f"# VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# GPU available:  True
# GPU name:       NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 2g.48gb
# VRAM:           50.9 GB


# 2. Verify DETR model builds correctly:

In [3]:
import argparse
from detr_detector.models import build_model

device = 'cuda' if torch.cuda.is_available() else "cpu"
args = argparse.Namespace(num_classes=2, # 1 class: bird, but they mentioned this value is n_classes+1
                          dataset_file='',
                          device=device,
                          lr_backbone=1e-5, # Learning rate of the backbone
                          enc_layers=6, # Number of encoder layers
                          dec_layers=6, # Number of decoder layers
                          pre_norm=False,
                          set_cost_class=1, # Class coefficient in the matching cost
                          set_cost_bbox=5, # L1 box coefficient in the matching cost
                          set_cost_giou=2, # giou box coefficient in the matching cost
                          bbox_loss_coef=5,
                          giou_loss_coef=2,
                          eos_coef=0.1, # Relative classification weight of the no-object class
                          hidden_dim=256,
                          nheads=8,
                          num_encoder_layers=6,
                          num_decoder_layers=6,
                          dim_feedforward=2048,
                          dropout=0.1,
                          backbone='resnet50',
                          dilation=False,
                          position_embedding='sine',
                          num_queries=100,
                          aux_loss=False,
                          masks=False
                    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, criterion, postprocessors = build_model(args)
model = model.to(device)
print(f"Model built successfully on {device}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

/home/hice1/wzhu97/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hice1/wzhu97/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model built successfully on cuda
Number of parameters: 41,501,895


# 3. Verify the forward pass works

## 3.1 Load H23 data (1% suebset)

In [4]:
# Load data use the code I tested before
from utilities.dataset import create_dataloaders

coco_fn = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23/Annotations/instances_merged.json'
fn_frame_list = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23_frame_list.subset_0.01.csv' 

# Split the data into train, test, validation
# Perform transformations: resizing, normalization
# num_workers: Enables multi-process data loading 
train_loader, val_loader, test_loader = create_dataloaders(frames_csv=fn_frame_list,
                                                           annotation_path=coco_fn, 
                                                           test_size=0.2, val_size=0.1,
                                                           batch_size=2, num_workers=4,
                                                           random_state=42)



# Frames with >1 annotations were excluded: N=32

# ##### Data split summary #####
# Split      Videos   Frames
# Train          60      195
# Val            18       57
# Test            9       27
# Total          87      279


In [5]:
# Check the Data loader worked as expected
images, annotations = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print(f"Input shape:         {images.shape}")                  # [2, 3, 720, 1280]
print(f"Pred logits shape:   {outputs['pred_logits'].shape}")  # [2, 100, 3]
print(f"Pred boxes shape:    {outputs['pred_boxes'].shape}")   # [2, 100, 4]

Input shape:         torch.Size([2, 3, 720, 1280])
Pred logits shape:   torch.Size([2, 100, 3])
Pred boxes shape:    torch.Size([2, 100, 4])


# 4. Set up the training loop (test run of one batch)

In [1]:
from utilities.coco_utils import coco_annos_to_detr_targets

# 1. check output shapes are as expected
print(f"Pred logits: {outputs['pred_logits'].shape}")   # should be [2, 100, 2]
print(f"Pred boxes:  {outputs['pred_boxes'].shape}")    # should be [2, 100, 4]

# 2. check criterion works with your annotations
loss_dict = criterion

(outputs, annotations)
print(f"Losses: {loss_dict}")

In [18]:
from utilities.coco_utils import coco_annos_to_detr_targets

images, annotations = next(iter(train_loader))
images  = images.to(device)

# The original annotations do not match DETR's expectation
# Need to convert to {'labels':[], 'boxes':[]}
targets = coco_annos_to_detr_targets(annotations, img_h=720, img_w=1280, device=device)

# verify format
print(f"# Num targets:          {len(targets)}")           # 2 (one per image)
print(f"# Target 0 labels:      {targets[0]['labels']}")   # tensor([0]) or tensor([])
print(f"# Target 0 boxes:       {targets[0]['boxes']}")    # tensor([[cx, cy, w, h]])

# now criterion works
with torch.no_grad():
    outputs   = model(images)
    loss_dict = criterion(outputs, targets)
print(f"# Losses: {loss_dict}")

# Num targets:          2
# Target 0 labels:      tensor([0], device='cuda:0')
# Target 0 boxes:       tensor([[0.7394, 0.4067, 0.0665, 0.0827]], device='cuda:0')
# Losses: {'loss_ce': tensor(1.6015, device='cuda:0'), 'class_error': tensor(0., device='cuda:0'), 'loss_bbox': tensor(1.1453, device='cuda:0'), 'loss_giou': tensor(0.9985, device='cuda:0'), 'cardinality_error': tensor(99., device='cuda:0')}


## An example example form the training

```
# Num targets:          2
# Target 0 labels:      tensor([0], device='cuda:0')
# Target 0 boxes:       tensor([[0.7394, 0.4067, 0.0665, 0.0827]], device='cuda:0')
# Losses: {'loss_ce': tensor(1.6015, device='cuda:0'), 'class_error': tensor(0., device='cuda:0'), 'loss_bbox': tensor(1.1453, device='cuda:0'), 'loss_giou': tensor(0.9985, device='cuda:0'), 'cardinality_error': tensor(99., device='cuda:0')}
```

### Bounding box breakdown
| Value | Meaning | Absolute pixels |
|---|---|---|
| `cx = 0.7394` | Bird center is 74% from left edge | ~946 px from left |
| `cy = 0.4067` | Bird center is 41% from top edge | ~293 px from top |
| `w  = 0.0665` | Bird width is 6.6% of frame width | ~85 px wide |
| `h  = 0.0827` | Bird height is 8.3% of frame height | ~60 px tall |

> Bird is relatively small (~85×60 px) in a 1280×720 frame, consistent with a bird detected in a video at distance.

### Losses interpretation (this is an untrained model)
| Loss | Value | Interpretation |
|---|---|---|
| `loss_ce` | 1.6015 | Normal for untrained model |
| `class_error` | 0.0 | No misclassification — only 1 class (bird) exists |
| `loss_bbox` | 1.1453 | Normal for untrained model |
| `loss_giou` | 0.9985 | Close to 1.0 — predicted boxes have no overlap with ground truth yet |
| `cardinality_error` | 99.0 | Model predicts 100 boxes but only 1 is correct — 99 false detections |

### Summary
All losses are expected to be high for an untrained model. The high `cardinality_error` 
of 99.0 is normal — DETR always outputs `num_queries=100` boxes before training, and it 
has not yet learned to suppress false detections. These values should decrease 
significantly as training progresses.


## DETR Loss Terms

| Loss | Meaning | Expected behavior after training |
|---|---|---|
| `loss_ce` | Cross entropy classification loss — penalizes wrong class predictions | Decreases |
| `class_error` | % of incorrectly classified predictions | Decreases toward 0 |
| `loss_bbox` | L1 loss between predicted and ground truth bounding box coordinates | Decreases |
| `loss_giou` | Generalized IoU loss — measures overlap between predicted and ground truth boxes. Close to 1.0 means no overlap | Decreases toward 0 |
| `cardinality_error` | Difference between number of predicted boxes and ground truth boxes. e.g. model predicts 99 extra boxes when only 1 bird exists | Decreases as model learns to suppress false detections |

## Notes
- All loss values are expected to be high for an **untrained model**
- `cardinality_error` is not used for backpropagation — it is a diagnostic metric only
- Total loss used for backpropagation = `loss_ce` + `loss_bbox` + `loss_giou` (weighted by coefficients in `args`)
- Coefficients in `args`: `bbox_loss_coef=5`, `giou_loss_coef=2`, `eos_coef=0.1`

# 5. Set up the training loop (full run)
Check the DETR example [here](https://github.com/facebookresearch/detr/blob/29901c51d7fe8712168b8d0d64351170bc0f83e0/main.py#L43).

## 5.1 Training and validation

In [8]:
import torch
import os
import json
import time
from pathlib import Path
from utilities.coco_utils import coco_annos_to_detr_targets
from utilities.evaluation import evaluate_map

def train_detr(model, criterion, train_loader, val_loader,
               device, img_h, img_w,
               num_epochs=10,
               lr=1e-4,
               lr_backbone=1e-5,
               output_dir='output/detr_test_run'):
    """
    Full training loop for DETR bird detection model.

    Args:
        model:        DETR model
        criterion:    DETR loss function (Hungarian matching + CE + bbox + giou)
        train_loader: DataLoader for training set
        val_loader:   DataLoader for validation set
        device:       torch device (cuda or cpu)
        img_h:        image height in pixels
        img_w:        image width in pixels
        num_epochs:   number of training epochs
        lr:           learning rate for transformer
        lr_backbone:  learning rate for ResNet50 backbone (smaller to preserve ImageNet weights)
        output_dir:   directory to save checkpoints and logs
    """
    # setup output directory and log file
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    log_path = output_dir / 'training_log.txt'

    def log(msg):
        """Print to notebook and save to log file simultaneously."""
        print(msg)
        with open(log_path, 'a') as f:
            f.write(msg + '\n')

    # set up optimizer with different learning rates for backbone vs transformer
    # backbone uses smaller lr to gently fine-tune ImageNet weights
    param_groups = [
        {'params': [p for n, p in model.named_parameters() 
                    if 'backbone' not in n and p.requires_grad], 
         'lr': lr},
        {'params': [p for n, p in model.named_parameters() 
                    if 'backbone' in n and p.requires_grad],
         'lr': lr_backbone}
    ]
    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

    # learning rate scheduler - reduces lr by 10x at epoch 7 (standard DETR schedule)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    # training state
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': []}

    log(f"{'='*60}")
    log(f"Training started: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    log(f"Epochs:           {num_epochs}")
    log(f"Train batches:    {len(train_loader)}")
    log(f"Val batches:      {len(val_loader)}")
    log(f"Device:           {device}")
    log(f"Output dir:       {output_dir}")
    log(f"{'='*60}")

    for epoch in range(num_epochs):
        epoch_start = time.time()

        # ── Training phase ──────────────────────────────────────
        model.train()
        criterion.train()
        train_losses = []

        for batch_idx, (images, annotations) in enumerate(train_loader):
            images  = images.to(device)
            targets = coco_annos_to_detr_targets(annotations, img_h, img_w, device)

            # forward pass
            outputs = model(images)
            loss_dict = criterion(outputs, targets)

            # total loss = weighted sum of all loss components
            # weights are defined in args (bbox_loss_coef, giou_loss_coef)
            losses = sum(loss_dict[k] * criterion.weight_dict[k] 
                        for k in loss_dict.keys() 
                        if k in criterion.weight_dict)

            # backward pass
            optimizer.zero_grad()
            losses.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)  # gradient clipping
            optimizer.step()

            train_losses.append(losses.item())

            # print batch progress every 10 batches
            if (batch_idx + 1) % 10 == 0:
                log(f"  Epoch [{epoch+1}/{num_epochs}] "
                    f"Batch [{batch_idx+1}/{len(train_loader)}] "
                    f"Loss: {losses.item():.4f}")

        avg_train_loss = sum(train_losses) / len(train_losses)

        # ── Validation phase ─────────────────────────────────────
        model.eval()
        criterion.eval()
        val_losses = []

        with torch.no_grad():
            for images, annotations in val_loader:
                images  = images.to(device)
                targets = coco_annos_to_detr_targets(annotations, img_h, img_w, device)

                outputs   = model(images)
                loss_dict = criterion(outputs, targets)
                losses    = sum(loss_dict[k] * criterion.weight_dict[k]
                               for k in loss_dict.keys()
                               if k in criterion.weight_dict)
                val_losses.append(losses.item())

        avg_val_loss = sum(val_losses) / len(val_losses)

        # update learning rate scheduler
        lr_scheduler.step()

        # track history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)

        # epoch summary
        epoch_time = time.time() - epoch_start
        log(f"\nEpoch [{epoch+1}/{num_epochs}] completed in {epoch_time:.1f}s")
        log(f"  Train loss: {avg_train_loss:.4f}")
        log(f"  Val loss:   {avg_val_loss:.4f}")
        log(f"  LR (transformer): {optimizer.param_groups[0]['lr']:.2e}")
        log(f"  LR (backbone):    {optimizer.param_groups[1]['lr']:.2e}")

        # save best model checkpoint
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            checkpoint = {
                'epoch':      epoch + 1,
                'model':      model.state_dict(),
                'optimizer':  optimizer.state_dict(),
                'scheduler':  lr_scheduler.state_dict(),
                'val_loss':   avg_val_loss,
                'train_loss': avg_train_loss,
                'args':       args
            }
            torch.save(checkpoint, output_dir / 'best_model.pth')
            log(f"  ✓ Best model saved (val_loss={avg_val_loss:.4f})")

        log('')   # blank line between epochs

    # save training history to json
    with open(output_dir / 'history.json', 'w') as f:
        json.dump(history, f, indent=2)

    log(f"{'='*60}")
    log(f"Training complete: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    log(f"Best val loss:     {best_val_loss:.4f}")
    log(f"Checkpoints:       {output_dir / 'best_model.pth'}")
    log(f"Log file:          {log_path}")
    log(f"History:           {output_dir / 'history.json'}")
    log(f"{'='*60}")

    return model, history

In [7]:
model, history = train_detr(
    model        = model,
    criterion    = criterion,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    img_h        = 720,
    img_w        = 1280,
    num_epochs   = 10,
    lr           = 1e-4,
    lr_backbone  = 1e-5,
    output_dir   = '/storage/ice1/2/7/wzhu97/2026_freeman_bird_behavior/output/detr_test_run'
)

Training started: 2026-04-16 13:23:27
Epochs:           10
Train batches:    98
Val batches:      29
Device:           cuda
Output dir:       /storage/ice1/2/7/wzhu97/2026_freeman_bird_behavior/output/detr_test_run
  Epoch [1/10] Batch [10/98] Loss: 5.4967
  Epoch [1/10] Batch [20/98] Loss: 4.8025
  Epoch [1/10] Batch [30/98] Loss: 3.8319
  Epoch [1/10] Batch [40/98] Loss: 0.0013
  Epoch [1/10] Batch [50/98] Loss: 0.0009
  Epoch [1/10] Batch [60/98] Loss: 1.6032
  Epoch [1/10] Batch [70/98] Loss: 2.2310
  Epoch [1/10] Batch [80/98] Loss: 1.4789
  Epoch [1/10] Batch [90/98] Loss: 0.0006

Epoch [1/10] completed in 7.9s
  Train loss: 2.5052
  Val loss:   2.2071
  LR (transformer): 1.00e-04
  LR (backbone):    1.00e-05
  ✓ Best model saved (val_loss=2.2071)

  Epoch [2/10] Batch [10/98] Loss: 2.8539
  Epoch [2/10] Batch [20/98] Loss: 3.8537
  Epoch [2/10] Batch [30/98] Loss: 1.3711
  Epoch [2/10] Batch [40/98] Loss: 3.2751
  Epoch [2/10] Batch [50/98] Loss: 4.7922
  Epoch [2/10] Batch [60/

## 5.2 Prediction (evaluation on test set)
* Start with IoU, mabybe also include mAP.
* Need a confusion matrix